In [1]:
import pandas as pd

# load initial raw data
raw_fridges = pd.read_csv('fridges.csv')
raw_stocking_log = pd.read_csv('stocking_log.csv')

In [2]:
# first pass inspection
raw_fridges

,f_id,nbhd,host,cap,inst,pwr
0,F03609,harlem,Green Grocer,"1,804",20240222,GRID
1,F04381,Crown Heights,the bakery,187,20220903,grid
2,F00020,jackson hts,community garden,504,20210126,Solar
3,F00195,Sunset Park,bodega grande,"1,744",20200826,GRID
4,F02113,harlem,The Bakery,"1,139",20241019,grid
...,...,...,...,...,...,...
4995,F03035,crown hts,the bakery,"1,197",20210522,solar
4996,F04289,Bushwick,green grocer,57,20200713,Generator
4997,F04531,Bushwick,Corner Deli,"1,319",20220403,Solar
4998,F04257,bushwick,Green Grocer,130,20230317,GRID


In [3]:
raw_stocking_log

,f_id,log_dt,temp,items,clean,restk
0,F01961,2025-01-15,33.0,11.0,Dirty,Yes
1,F04734,2020-05-24,37.0,5.0,needs cleaning,YES
2,F01340,2025-07-11,42.0,13.0,Clean,YES
3,F00118,2024-12-14,38.0,6.0,dirty,y
4,F01390,2020-02-03,37.0,41.0,Clean,No
...,...,...,...,...,...,...
4995,F01231,2023-11-25,33.0,32.0,clean,no
4996,F00169,2020-12-06,43.0,8.0,NaN,n
4997,F00210,2020-06-14,44.0,46.0,OK,Yes
4998,F03639,2020-08-25,45.0,10.0,Dirty,y


it seems we can combine the datasets based on the f_id

In [4]:
df = raw_fridges.merge(raw_stocking_log, on="f_id")
df

,f_id,nbhd,host,cap,inst,pwr,log_dt,temp,items,clean,restk
0,F03609,harlem,Green Grocer,"1,804",20240222,GRID,2021-11-28,NaN,9.0,needs cleaning,yes
1,F04381,Crown Heights,the bakery,187,20220903,grid,2025-10-01,37.0,47.0,CLEAN,n
2,F00020,jackson hts,community garden,504,20210126,Solar,2024-02-14,35.0,34.0,Dirty,y
3,F00195,Sunset Park,bodega grande,"1,744",20200826,GRID,2021-01-12,999.0,57.0,CLEAN,Yes
4,F02113,harlem,The Bakery,"1,139",20241019,grid,2024-08-03,34.0,29.0,needs cleaning,YES
...,...,...,...,...,...,...,...,...,...,...,...
4995,F03035,crown hts,the bakery,"1,197",20210522,solar,2022-10-19,45.0,7.0,needs cleaning,NO
4996,F04289,Bushwick,green grocer,57,20200713,Generator,2025-01-10,37.0,47.0,Dirty,YES
4997,F04531,Bushwick,Corner Deli,"1,319",20220403,Solar,2025-06-24,38.0,47.0,dirty,no
4998,F04257,bushwick,Green Grocer,130,20230317,GRID,2023-12-20,37.0,17.0,CLEAN,YES


In [5]:
# let's double check if there are any records missing an f_id. this would be a serious gap in the data.
df['f_id'].isna().value_counts()

f_id
False    5000
Name: count, dtype: int64

In [6]:
# ok, all rows have a f_id

In [7]:
# let's investigate the clean column, since a fridge's cleanliness heavily affects its usage and effectiveness
df['clean'].value_counts()

clean
 ok               624
OK                623
Dirty             619
Clean             600
CLEAN             599
needs cleaning    581
clean             578
dirty             572
Name: count, dtype: int64

In [14]:
# definitely looks like it needs to be normalized
df['clean_normalized'] = (df['clean']
    .str.lower() # lowercase everything first
    .replace({'needs cleaning': 'dirty'}) # reduce number of possible values
    .str.strip() # remove any starting or trailing whitespace
)
df['clean_normalized']

0       dirty
1       clean
2       dirty
3       clean
4       dirty
        ...  
4995    dirty
4996    dirty
4997    dirty
4998    clean
4999       ok
Name: clean_normalized, Length: 5000, dtype: str

In [15]:
# ok, let's do some analysis on this. but to do so, we might need to clean it more
df['nbhd'].unique()

<StringArray>
[            'harlem',      'Crown Heights',        'jackson hts',
        'Sunset Park',             'Harlem',           'Bed-Stuy',
           'Flatbush',          'crown hts',         'mott haven',
           'bushwick',        'sunset park',         'Mott Haven',
    'Jackson Heights',           'Bushwick', 'Bedford-Stuyvesant',
          'ridgewood',          'flatbush ',          'Ridgewood',
           'bed-stuy',         ' Bushwick ']
Length: 20, dtype: str

In [17]:
df['neighborhood'] = (df['nbhd']
    .str.lower()
    .replace({
        "bed-stuy": "bedford-stuyvesant",
        "crown hts": "crown heights",
        "jackson hts": "jackson heights"
    })
    .str.strip()
)
df['neighborhood'].unique()

<StringArray>
[            'harlem',      'crown heights',    'jackson heights',
        'sunset park', 'bedford-stuyvesant',           'flatbush',
         'mott haven',           'bushwick',          'ridgewood']
Length: 9, dtype: str

In [37]:
# now let's investigate cleanliness per neighborhood!
(df.groupby('neighborhood')['clean_normalized'] # groupby allows us to do sub-aggregates, based on a specific column
    .value_counts(normalize=True) # normalize replaces absolute counts with decimal values representing percentage
    .sort_index() # by default, value_counts sorts by value. to make this easier to read, sort by the clean_normalized column so structure is easier to read
    .to_frame().style.format("{:.1%}") # convert the decimal from value_counts to percentage
)

In [46]:
# looks great but let's just pull out the dirty ones
# now let's investigate cleanliness per neighborhood!
(df[df['clean_normalized'] == 'dirty']
    .groupby('neighborhood')['clean_normalized'] # groupby allows us to do sub-aggregates, based on a specific column
    .value_counts()
)

neighborhood        clean_normalized
bedford-stuyvesant  dirty               271
bushwick            dirty               263
crown heights       dirty               207
flatbush            dirty               161
harlem              dirty               175
jackson heights     dirty               184
mott haven          dirty               165
ridgewood           dirty               171
sunset park         dirty               175
Name: count, dtype: int64

In [59]:
# and now by percent dirty
(df.groupby('neighborhood')['clean_normalized'] # groupby allows us to do sub-aggregates, based on a specific column
    .value_counts(normalize=True) # normalize replaces absolute counts with decimal values representing percentage
    .xs('dirty', level='clean_normalized') # extract only the dirty percentage counts
    .to_frame()
    .sort_values(by='proportion', ascending=False) # sort them by proportion
    .style.format("{:.1%}") # convert the decimal from value_counts to percentage
    
)

,proportion
neighborhood,
crown heights,42.6%
jackson heights,38.7%
ridgewood,38.0%
bedford-stuyvesant,36.9%
harlem,36.2%
sunset park,36.1%
mott haven,35.7%
bushwick,35.3%
flatbush,34.0%
